In [0]:
import sys
sys.path.append("../lib")
import utils
import delta

dbutils.widgets.text("tabela", "customers")
dbutils.widgets.text("chave_merge", "customer_id")

tabela = dbutils.widgets.get("tabela")
chave_merge = dbutils.widgets.get("chave_merge")

catalog = "projeto_olist"
schema = "bronze"

In [0]:
if not utils.table_exists(spark, catalog, schema, f"{tabela}_fullload"):
    df_full = spark.read.format("parquet").load(f"/Volumes/projeto_olist/olist/fullload/{tabela}_fullload/")

    (df_full.coalesce(1)
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{catalog}.{schema}.{tabela}_fullload"))

In [0]:
data_schema = utils.import_schema(tabela)

df_stream = (spark.readStream
                  .format("cloudFiles")
                  .option("cloudFiles.format", "parquet")
                  .schema(data_schema)
                  .load(f"/Volumes/projeto_olist/olist/cdc/{tabela}/"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

bronze = delta.DeltaTable.forName(spark, f"{catalog}.{schema}.{tabela}_fullload")

def upsert(df, batchId):
    janela = Window.partitionBy(chave_merge).orderBy(F.col("data_particao").desc())

    df_unique = (df.withColumn("rn", F.row_number().over(janela))
                   .filter("rn = 1")
                   .drop("rn"))

    (bronze.alias("b")
           .merge(df_unique.alias("d"), f"b.{chave_merge} = d.{chave_merge}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())

In [0]:
stream = (df_stream.writeStream
                   .option("checkpointLocation", f"/Volumes/projeto_olist/olist/cdc/{tabela}_checkpoint/")
                   .foreachBatch(upsert)
                   .trigger(availableNow=True))

stream.start()

In [0]:
display(spark.sql(f"select * from {catalog}.{schema}.{tabela}_fullload"))